# Handling Parquet Files with Python

**Parquet** is a **columnar** file format: instead of writing the data row by row (like CSV or Excel), it writes all the values of a column together. In this notebook we compare Parquet with CSV and Excel in terms of file size and speed.

> Don't confuse columnar *formats* like Parquet with *column-family* databases like Cassandra: see [Lesson 3](https://github.com/Naviden/Introduction-to-Big-Data-and-NoSQL/blob/main/lessons/03_sql_vs_nosql.md).

In [ ]:
import os
import time
import numpy as np
import pandas as pd

## Generate sample data

In [ ]:
np.random.seed(42)
n = 100_000
data = {
    'id': np.arange(1, n + 1),
    'name': np.random.choice(['Alice', 'Bob', 'Charlie', 'David', 'Eva'], n),
    'surname': np.random.choice(['Smith', 'Johnson', 'Williams', 'Jones', 'Brown'], n),
    'email': [f'user{i}@example.com' for i in range(1, n + 1)],
    'address': np.random.choice(['123 Main St', '456 Oak St', '789 Pine St', '101 Maple St'], n),
    'age': np.random.randint(18, 80, n),
}

# Create DataFrame
df = pd.DataFrame(data)

In [ ]:
df.sample(10)

## Save the same data as Excel, CSV and Parquet

In [ ]:
df.to_excel('users_info.xlsx', index=False)
df.to_csv('users_info.csv', index=False)
df.to_parquet('users_info.parquet', index=False)

## Compare file size and loading time
Note that we measure the size of the **files on disk** with `os.path.getsize`. Once loaded in pandas, the three DataFrames are identical and use the same memory.

In [ ]:
# Read each file once before timing, so that importing the libraries (openpyxl, pyarrow) is not counted
pd.read_excel('users_info.xlsx', nrows=5)
pd.read_csv('users_info.csv', nrows=5)
pd.read_parquet('users_info.parquet', columns=['id'])

results = []
for path, reader in [('users_info.xlsx', pd.read_excel),
                     ('users_info.csv', pd.read_csv),
                     ('users_info.parquet', pd.read_parquet)]:
    start = time.perf_counter()
    reader(path)
    elapsed = time.perf_counter() - start
    results.append({'file': path,
                    'size_MB': round(os.path.getsize(path) / 1024**2, 2),
                    'load_seconds': round(elapsed, 3)})

pd.DataFrame(results)

# Advantages of Columnar Formats Like Parquet

Columnar formats such as Parquet store data by columns instead of rows. This provides several advantages for analytical tasks:

- **Faster Queries**: When only specific columns are needed, columnar formats read only those columns, reducing I/O.
- **Efficient Compression**: Similar values sit next to each other in a column, so they compress much better (look at the `name` column: only 5 different values in 100,000 rows).
- **Schema Included**: The file stores the type of each column, so `age` comes back as a number without any guessing.
- **Improved Performance in Distributed Systems**: Optimized for big data frameworks like Spark, which can split a Parquet file across many machines.

## Reading only the columns you need
With CSV, pandas has to read every line completely even if you only want one column. With Parquet, only the requested columns are read from disk.

In [ ]:
import timeit

csv_time = timeit.timeit(lambda: pd.read_csv('users_info.csv', usecols=['age']), number=5) / 5
parquet_time = timeit.timeit(lambda: pd.read_parquet('users_info.parquet', columns=['age']), number=5) / 5

print(f"CSV, one column:     {csv_time:.4f} s")
print(f"Parquet, one column: {parquet_time:.4f} s")

## Query time comparison

In [ ]:
# Load the file and filter it, 5 times each
def query(reader, path):
    df = reader(path)
    return df[df['name'] == 'Alice']

query_times = {
    'Excel': timeit.timeit(lambda: query(pd.read_excel, 'users_info.xlsx'), number=5) / 5,
    'CSV': timeit.timeit(lambda: query(pd.read_csv, 'users_info.csv'), number=5) / 5,
    'Parquet': timeit.timeit(lambda: query(pd.read_parquet, 'users_info.parquet'), number=5) / 5,
}
query_times

In [ ]:
import matplotlib.pyplot as plt

# Visualize time comparison
plt.bar(list(query_times.keys()), list(query_times.values()))
plt.ylabel('Time (s)')
plt.title('Load + filter time')
plt.show()

## Exercises
1. Save the DataFrame as Parquet with `compression='gzip'` and with `compression=None`. Compare the file sizes.
2. Repeat the size comparison with `n = 1_000_000` (skip Excel, it would be very slow). How do the differences change?
3. Why would a columnar format be a bad choice for an application that often needs to read or update a single complete row?